Next we’ll build item-based collaborative filtering: a personalized recommender that identifies movies with similar patterns of user feedback, then recommends movies related to a user’s previous likes.

For our first version, “liked” will mean a training rating of 4 stars or higher. Users without positive training history will receive popularity recommendations as a fallback.

First we will reproduce the same training, validation, and testing data sets as before. Reproducing the same split lets us compare approaches under the same evaluation conditions. The new model’s similarities will be calculated from training data.

In [2]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data" / "raw" / "ml-latest-small"

movies = pd.read_csv(data_dir / "movies.csv")
ratings = pd.read_csv(data_dir / "ratings.csv")

ratings["rated_at"] = pd.to_datetime(
    ratings["timestamp"],
    unit="s",
    utc=True
)

dates = ratings["rated_at"]

train_cutoff = dates.quantile(0.60)
validation_cutoff = dates.quantile(0.80)

train_ratings = ratings.loc[
    dates <= train_cutoff
].copy()

validation_ratings = ratings.loc[
    (dates > train_cutoff) & (dates <= validation_cutoff)
].copy()

test_ratings = ratings.loc[
    dates > validation_cutoff
].copy()

print("Training rows:", len(train_ratings))
print("Validation rows:", len(validation_ratings))
print("Test rows:", len(test_ratings))

Training rows: 60502
Validation rows: 20167
Test rows: 20167


In [3]:
training_feedback = train_ratings[
    ["userId", "movieId", "rating"]
].copy()

training_feedback["liked"] = (
    training_feedback["rating"] >= 4.0
).astype(int)

user_movie_matrix = (
    training_feedback.pivot(
        index="userId",
        columns="movieId",
        values="liked"
    )
    .fillna(0)
    .astype(int)
)

print("Users:", user_movie_matrix.shape[0])
print("Movies:", user_movie_matrix.shape[1])

print(
    "Positive training ratings:",
    training_feedback["liked"].sum()
)

user_movie_matrix.iloc[:5, :8]

Users: 381
Movies: 6233
Positive training ratings: 29115


movieId,1,2,3,4,5,6,7,8
userId,,,,,,,,
1,1,0,1,0,0,1,0,0
4,0,0,0,0,0,0,0,0
5,1,0,0,0,0,0,0,0
6,0,1,1,0,1,1,1,0
7,1,0,0,0,0,0,0,0


Next, we’ll calculate cosine similarity between two movies. This measures how much their patterns of positive feedback overlap—a standard approach in item-based collaborative filtering. files.grouplens.org

For our binary feedback:
\[
\text{Similarity}(A,B)
=
\frac{\text{Users who liked both movies}}
{\sqrt{\text{Users who liked A}\times\text{Users who liked B}}}
\]
The denominator normalizes the shared-user count using each movie’s number of likes.

Let’s compare Toy Story (movieId=1) and Jumanji (movieId=2).

In [4]:
import numpy as np

toy_story_likes = user_movie_matrix[1]
jumanji_likes = user_movie_matrix[2]

toy_story_like_count = toy_story_likes.sum()
jumanji_like_count = jumanji_likes.sum()

shared_like_count = (
    toy_story_likes * jumanji_likes
).sum()

movie_similarity_score = (
    shared_like_count
    / np.sqrt(toy_story_like_count * jumanji_like_count)
)

print("Users who liked Toy Story:", toy_story_like_count)
print("Users who liked Jumanji:", jumanji_like_count)
print("Users who liked both:", shared_like_count)
print("Cosine similarity:", movie_similarity_score)

Users who liked Toy Story: 104
Users who liked Jumanji: 36
Users who liked both: 16
Cosine similarity: 0.26148818018424536


Next, we’ll calculate Toy Story’s similarity to every training movie, using scikit-learn’s implementation of the same calculation.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

movie_user_matrix = user_movie_matrix.T

toy_story_scores = cosine_similarity(
    movie_user_matrix.loc[[1]],
    movie_user_matrix
)[0]

similar_movies = pd.DataFrame({
    "movieId": movie_user_matrix.index,
    "similarity": toy_story_scores
})

similar_movies = similar_movies.merge(
    movies[["movieId", "title"]],
    on="movieId",
    how="left",
    validate="one_to_one"
)

similar_movies = similar_movies.loc[
    similar_movies["movieId"] != 1
].sort_values(
    ["similarity", "movieId"],
    ascending=[False, True]
)

similar_movies[
    ["movieId", "title", "similarity"]
].head(10).reset_index(drop=True)

,movieId,title,similarity
0,3114,Toy Story 2 (1999),0.502459
1,260,Star Wars: Episode IV - A New Hope (1977),0.461166
2,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),0.450360
3,318,"Shawshank Redemption, The (1994)",0.445399
4,480,Jurassic Park (1993),0.444208
5,588,Aladdin (1992),0.433148
6,1198,Raiders of the Lost Ark (Indiana Jones and the...,0.432692
7,2762,"Sixth Sense, The (1999)",0.430265
8,1196,Star Wars: Episode V - The Empire Strikes Back...,0.426583
9,589,Terminator 2: Judgment Day (1991),0.424328


Score each candidate movie by its average similarity to the movies the user liked.

In [6]:
example_user_id = 1

user_history = train_ratings.loc[
    train_ratings["userId"] == example_user_id
]

user_liked_movie_ids = user_history.loc[
    user_history["rating"] >= 4.0,
    "movieId"
]

similarities_to_liked = cosine_similarity(
    movie_user_matrix.loc[user_liked_movie_ids],
    movie_user_matrix
)

user_1_scores = pd.DataFrame({
    "movieId": movie_user_matrix.index,
    "score": similarities_to_liked.mean(axis=0)
})

print("Movies rated in training:", len(user_history))
print("Movies liked in training:", len(user_liked_movie_ids))
print("Similarity array shape:", similarities_to_liked.shape)

Movies rated in training: 232
Movies liked in training: 200
Similarity array shape: (200, 6233)


Remove previously rated movies and display recommendations

In [7]:
user_1_recommendations = user_1_scores.loc[
    ~user_1_scores["movieId"].isin(user_history["movieId"])
].merge(
    movies[["movieId", "title"]],
    on="movieId",
    how="left",
    validate="one_to_one"
).sort_values(
    ["score", "movieId"],
    ascending=[False, True]
).head(10).reset_index(drop=True)

user_1_recommendations[
    ["movieId", "title", "score"]
]

,movieId,title,score
0,2918,Ferris Bueller's Day Off (1986),0.248263
1,858,"Godfather, The (1972)",0.239522
2,1221,"Godfather: Part II, The (1974)",0.229416
3,1036,Die Hard (1988),0.228957
4,541,Blade Runner (1982),0.227091
5,1200,Aliens (1986),0.224153
6,2762,"Sixth Sense, The (1999)",0.222821
7,589,Terminator 2: Judgment Day (1991),0.217092
8,924,2001: A Space Odyssey (1968),0.214593
9,1641,"Full Monty, The (1997)",0.213656


We will assess the performance of these personalized reccomendations using the validation data. We will recreate the popularity ranking function to use as fallback reccomendations if the user has no records from which to build a profile

In [8]:
movie_popularity = (
    train_ratings.groupby("movieId")
    .size()
    .reset_index(name="training_rating_count")
    .merge(
        movies[["movieId", "title"]],
        on="movieId",
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        ["training_rating_count", "movieId"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

movie_popularity.head(10)

,movieId,training_rating_count,title
0,296,201,Pulp Fiction (1994)
1,356,193,Forrest Gump (1994)
2,480,185,Jurassic Park (1993)
3,318,177,"Shawshank Redemption, The (1994)"
4,593,177,"Silence of the Lambs, The (1991)"
5,110,174,Braveheart (1995)
6,589,166,Terminator 2: Judgment Day (1991)
7,457,160,"Fugitive, The (1993)"
8,592,160,Batman (1989)
9,780,157,Independence Day (a.k.a. ID4) (1996)


In [9]:
def recommend_item_based(
    user_id,
    training_data,
    movie_user_matrix,
    popularity_ranking,
    n=10
):
    """Recommend using mean item similarity, with popularity as a fallback."""

    user_history = training_data.loc[
        training_data["userId"] == user_id
    ]

    liked_movie_ids = user_history.loc[
        user_history["rating"] >= 4.0,
        "movieId"
    ]

    recommendations = popularity_ranking.loc[
        ~popularity_ranking["movieId"].isin(user_history["movieId"])
    ].copy()

    recommendations["score"] = 0.0

    if not liked_movie_ids.empty:
        similarities = cosine_similarity(
            movie_user_matrix.loc[liked_movie_ids],
            movie_user_matrix
        )

        scores = pd.Series(
            similarities.mean(axis=0),
            index=movie_user_matrix.index
        )

        recommendations["score"] = (
            recommendations["movieId"].map(scores).fillna(0.0)
        )

    return recommendations.sort_values(
        ["score", "training_rating_count", "movieId"],
        ascending=[False, False, True]
    ).head(n).reset_index(drop=True)

In [10]:
for example_user_id in [1, 2]:
    print("Recommendations for user:", example_user_id)

    display(
        recommend_item_based(
            user_id=example_user_id,
            training_data=train_ratings,
            movie_user_matrix=movie_user_matrix,
            popularity_ranking=movie_popularity,
            n=10
        )
    )

Recommendations for user: 1


,movieId,training_rating_count,title,score
0,2918,71,Ferris Bueller's Day Off (1986),0.248263
1,858,106,"Godfather, The (1972)",0.239522
2,1221,78,"Godfather: Part II, The (1974)",0.229416
3,1036,90,Die Hard (1988),0.228957
4,541,82,Blade Runner (1982),0.227091
5,1200,85,Aliens (1986),0.224153
6,2762,108,"Sixth Sense, The (1999)",0.222821
7,589,166,Terminator 2: Judgment Day (1991),0.217092
8,924,73,2001: A Space Odyssey (1968),0.214593
9,1641,49,"Full Monty, The (1997)",0.213656


Recommendations for user: 2


,movieId,training_rating_count,title,score
0,296,201,Pulp Fiction (1994),0.0
1,356,193,Forrest Gump (1994),0.0
2,480,185,Jurassic Park (1993),0.0
3,318,177,"Shawshank Redemption, The (1994)",0.0
4,593,177,"Silence of the Lambs, The (1991)",0.0
5,110,174,Braveheart (1995),0.0
6,589,166,Terminator 2: Judgment Day (1991),0.0
7,457,160,"Fugitive, The (1993)",0.0
8,592,160,Batman (1989),0.0
9,780,157,Independence Day (a.k.a. ID4) (1996),0.0


In [11]:
validation_positives = validation_ratings.loc[
    validation_ratings["rating"] >= 4.0
].copy()

evaluation_user_ids = validation_positives["userId"].unique()

item_based_rows = []

for user_id in evaluation_user_ids:
    user_positive_movie_ids = validation_positives.loc[
        validation_positives["userId"] == user_id,
        "movieId"
    ]

    recommendations = recommend_item_based(
        user_id=user_id,
        training_data=train_ratings,
        movie_user_matrix=movie_user_matrix,
        popularity_ranking=movie_popularity,
        n=10
    )

    hits = recommendations["movieId"].isin(
        user_positive_movie_ids
    ).sum()

    item_based_rows.append({
        "userId": user_id,
        "positive_movie_count": len(user_positive_movie_ids),
        "hits_at_10": hits,
        "recall_at_10": hits / len(user_positive_movie_ids)
    })

item_based_results = pd.DataFrame(item_based_rows)

print("Users evaluated:", len(item_based_results))

print(
    "Mean Recall@10:",
    item_based_results["recall_at_10"].mean()
)

item_based_results.head()

Users evaluated: 165
Mean Recall@10: 0.04476194860303786


,userId,positive_movie_count,hits_at_10,recall_at_10
0,2,19,0,0.000000
1,3,16,0,0.000000
2,10,50,0,0.000000
3,15,6,0,0.000000
4,16,53,1,0.018868


The item-based recommender improved overall validation Recall@10.
Approach	Mean Recall@10
Popularity baseline	3.98%
Item-based with popularity fallback	4.48%


That is an increase of approximately 0.50 percentage points, or 12.6% relative to the baseline.
This is an observed improvement under our current validation setup. We’ll reserve the test set for the final assessment.
Next, let’s see which users account for the improvement. Run:

In [12]:
item_based_results["has_training_history"] = (
    item_based_results["userId"].isin(
        train_ratings["userId"]
    )
)

item_based_group_summary = (
    item_based_results.groupby("has_training_history")
    .agg(
        users=("userId", "size"),
        mean_recall_at_10=("recall_at_10", "mean")
    )
)

item_based_group_summary

,users,mean_recall_at_10
has_training_history,,
False,140,0.041847
True,25,0.061086


Personalization improved mean recall for users with training history by 3.31 percentage points.
Group	Users	Popularity baseline	Item-based recommender
No training history	140	4.18%	4.18%
Has training history	25	2.80%	6.11%
Overall	165	3.98%	4.48%


The returning-user average more than doubled, although it started from a low baseline. The unchanged new-user result is expected because those users receive the popularity fallback.
Next, let’s check how many individual users improved. An average can increase because of large gains for a few people.

In [13]:
returning_results = item_based_results.loc[
    item_based_results["has_training_history"]
].copy()

baseline_recalls = {}

for user_id in returning_results["userId"]:
    rated_ids = train_ratings.loc[
        train_ratings["userId"] == user_id,
        "movieId"
    ]

    positive_ids = validation_positives.loc[
        validation_positives["userId"] == user_id,
        "movieId"
    ]

    popular_ids = movie_popularity.loc[
        ~movie_popularity["movieId"].isin(rated_ids),
        "movieId"
    ].head(10)

    baseline_recalls[user_id] = (
        popular_ids.isin(positive_ids).sum()
        / len(positive_ids)
    )

returning_results["popularity_recall_at_10"] = (
    returning_results["userId"].map(baseline_recalls)
)

returning_results["recall_change"] = (
    returning_results["recall_at_10"] 
    - returning_results["popularity_recall_at_10"]
)

print("Users improved:", (returning_results["recall_change"] > 0).sum())
print("Users worsened:", (returning_results["recall_change"] < 0).sum())
print("Users unchanged:", (returning_results["recall_change"] == 0).sum())

Users improved: 11
Users worsened: 5
Users unchanged: 9


## Item-based collaborative filtering: validation findings

### Approach

This recommender uses training ratings of 4.0 or higher as positive
feedback. Movies are compared using cosine similarity between their
user feedback vectors.

For each user, candidate movies are scored by their average similarity
to movies the user liked during training. Movies already rated by that
user are excluded. Users without positive training history receive
popularity-based recommendations.

### Evaluation

Both methods were evaluated using the same chronological split,
candidate movies, and definition of positive feedback. Recall@10 was
calculated for each user with at least one positive validation rating,
then averaged across users.

| User group | Users | Popularity Recall@10 | Item-based Recall@10 |
|---|---:|---:|---:|
| No training history | 140 | 4.18% | 4.18% |
| Has training history | 25 | 2.80% | 6.11% |
| Overall | 165 | 3.98% | 4.48% |

Among the 25 users with training history, recall improved for 11 users,
worsened for 5, and stayed unchanged for 9.

### Interpretation and limitations

Item-based recommendations increased mean Recall@10 by approximately
3.31 percentage points for users with training history. The overall
increase was smaller, approximately 0.50 percentage points, because
140 of the 165 evaluated users had no training history and received
the same popularity fallback.

The recommender still cannot recommend movies absent from its
training candidate set. These positive validation movies remain in
the recall denominator.

The results show an improvement for this validation cohort, but the
returning-user group contains only 25 users. Test-set performance
has not yet been evaluated.

In [15]:
item_based_results = pd.DataFrame(item_based_results)

from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

item_based_results.to_csv(
    processed_dir / "item_based_validation_results.csv",
    index=False
)

print("Saved collaborative-filtering results")

Saved collaborative-filtering results
